# DeepGCNRT Results Board

This notebook has two goals:

1. inspect a single run directory (`log.csv` + `test_predictions.csv`);
2. inspect transfer-learning results with the **same aggregation rule used by the official DeepGCN-RT release**, namely a simple mean over all recorded fold/seed rows in `summary.csv`.

For the current MiniDataset experiments, that means the displayed headline score is the mean over the 10 fold-level test metrics stored in each dataset-level `summary.csv`.

In [ ]:
from pathlib import Path
import csv
import math
import statistics
import zipfile
import xml.etree.ElementTree as ET
from pprint import pprint

import matplotlib.pyplot as plt

try:
    import pandas as pd
except Exception:
    pd = None

plt.style.use("seaborn-v0_8-whitegrid")


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "Train").exists() and (candidate / "Reference").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current working directory.")


REPO_ROOT = find_repo_root()
RESULT_ROOT = REPO_ROOT / "Train" / "Results" / "DeepGCNRT"
OFFICIAL_SUMMARY_XLSX = REPO_ROOT / "Reference" / "DeepGCN-RT" / "result" / "result_summary.xlsx"

PAPER_SEVEN = [
    "Eawag_XBridgeC18_364",
    "FEM_lipids_72",
    "FEM_long_412",
    "IPB_Halle_82",
    "LIFE_new_184",
    "LIFE_old_194",
    "UniToyama_Atlantis_143",
]

EXTRA_THREE = [
    "Cao_HILIC_116",
    "FEM_short_73",
    "MTBLS87_147",
]

TRANSFER_ORDER = PAPER_SEVEN + EXTRA_THREE


def timestamp_dirs(root):
    root = Path(root)
    if not root.exists():
        return []
    return sorted(path for path in root.iterdir() if path.is_dir() and path.name[:4].isdigit())


def latest_stage_run(stage_name):
    runs = timestamp_dirs(RESULT_ROOT / stage_name)
    if not runs:
        raise FileNotFoundError(f"No timestamped runs found under {RESULT_ROOT / stage_name}")
    return runs[-1]


def read_csv_rows(path):
    with Path(path).open(newline="") as f:
        return list(csv.DictReader(f))


def read_predictions(path):
    rows = read_csv_rows(path)
    y = [float(row["target"]) for row in rows]
    p = [float(row["prediction"]) for row in rows]
    return y, p


def compute_metrics(y, p):
    n = len(y)
    abs_err = [abs(a - b) for a, b in zip(y, p)]
    sq_err = [(a - b) ** 2 for a, b in zip(y, p)]
    y_mean = sum(y) / n
    denom = sum((value - y_mean) ** 2 for value in y)
    abs_sorted = sorted(abs_err)
    medae = abs_sorted[n // 2] if n % 2 else 0.5 * (abs_sorted[n // 2 - 1] + abs_sorted[n // 2])
    mse = sum(sq_err) / n
    return {
        "n": n,
        "mae": sum(abs_err) / n,
        "medae": medae,
        "mape": sum(abs(a - b) / max(abs(a), 1e-8) for a, b in zip(y, p)) / n,
        "mse": mse,
        "rmse": math.sqrt(mse),
        "r2": 0.0 if denom <= 0 else 1.0 - sum(sq_err) / denom,
    }


def parse_scalar(value):
    if value in (None, ""):
        return None
    try:
        if any(token in str(value) for token in [".", "e", "E"]):
            return float(value)
        return int(value)
    except ValueError:
        return value


def normalize_rows(rows):
    normalized = []
    for row in rows:
        normalized.append({key: parse_scalar(value) for key, value in row.items()})
    return normalized


def read_xlsx_first_sheet(path):
    ns = {"a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    with zipfile.ZipFile(path) as zf:
        shared_strings = []
        if "xl/sharedStrings.xml" in zf.namelist():
            root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
            for item in root.findall("a:si", ns):
                parts = [node.text or "" for node in item.findall(".//a:t", ns)]
                shared_strings.append("".join(parts))

        sheet = ET.fromstring(zf.read("xl/worksheets/sheet1.xml"))
        rows = []
        for row in sheet.findall(".//a:sheetData/a:row", ns):
            values = []
            for cell in row.findall("a:c", ns):
                cell_type = cell.get("t")
                value = cell.find("a:v", ns)
                if value is None:
                    values.append("")
                elif cell_type == "s":
                    values.append(shared_strings[int(value.text)])
                else:
                    values.append(value.text)
            rows.append(values)
        return rows


OFFICIAL_TRANSFER_MAE = {}
for row in read_xlsx_first_sheet(OFFICIAL_SUMMARY_XLSX)[1:]:
    dataset_name = row[0].replace("_result", "")
    OFFICIAL_TRANSFER_MAE[dataset_name] = float(row[1])


def candidate_dataset_dirs(dataset_name):
    candidates = [
        RESULT_ROOT / "MiniDatasets" / dataset_name,
        RESULT_ROOT / "RIKENMONA" / dataset_name,
    ]
    matches = [path for path in candidates if path.exists()]
    if matches:
        return matches

    fallback = []
    for path in RESULT_ROOT.rglob(dataset_name):
        if path.is_dir() and (path / "summary.csv").exists():
            fallback.append(path)
    return sorted(fallback)


def find_dataset_dir(dataset_name):
    matches = candidate_dataset_dirs(dataset_name)
    if not matches:
        raise FileNotFoundError(f"Could not find a results directory for dataset {dataset_name}")
    for path in matches:
        if path.parent.name == "MiniDatasets":
            return path
    return matches[0]


def load_dataset_summary(dataset_name):
    dataset_dir = find_dataset_dir(dataset_name)
    summary_path = dataset_dir / "summary.csv"
    if summary_path.exists():
        rows = normalize_rows(read_csv_rows(summary_path))
        return dataset_dir, rows

    rows = []
    for run_dir in timestamp_dirs(dataset_dir):
        y, p = read_predictions(run_dir / "test_predictions.csv")
        metrics = compute_metrics(y, p)
        rows.append(
            {
                "seed": None,
                "fold": None,
                "num_folds": None,
                "best_epoch": None,
                "best_valid_loss": None,
                "test_mae": metrics["mae"],
                "test_medae": metrics["medae"],
                "test_mape": metrics["mape"],
                "test_mse": metrics["mse"],
                "test_rmse": metrics["rmse"],
                "test_r2": metrics["r2"],
                "result_dir": str(run_dir.relative_to(REPO_ROOT)),
            }
        )
    return dataset_dir, rows


def aggregate_summary_rows(rows):
    metric_keys = [
        "test_mae",
        "test_medae",
        "test_mape",
        "test_mse",
        "test_rmse",
        "test_r2",
    ]
    aggregate = {"num_rows": len(rows)}
    for key in metric_keys:
        values = [float(row[key]) for row in rows if row.get(key) is not None]
        aggregate[f"mean_{key}"] = sum(values) / len(values)
        aggregate[f"std_{key}"] = statistics.stdev(values) if len(values) > 1 else 0.0

    best_epochs = [int(row["best_epoch"]) for row in rows if row.get("best_epoch") is not None]
    aggregate["min_best_epoch"] = min(best_epochs) if best_epochs else None
    aggregate["max_best_epoch"] = max(best_epochs) if best_epochs else None

    test_sizes = []
    for row in rows:
        result_dir = row.get("result_dir")
        if not result_dir:
            continue
        run_dir = Path(result_dir)
        if not run_dir.is_absolute():
            run_dir = REPO_ROOT / run_dir
        pred_path = run_dir / "test_predictions.csv"
        if pred_path.exists():
            test_sizes.append(len(read_predictions(pred_path)[0]))
    aggregate["mean_test_size"] = sum(test_sizes) / len(test_sizes) if test_sizes else None
    return aggregate


def build_transfer_overview(dataset_names=None):
    dataset_names = TRANSFER_ORDER if dataset_names is None else dataset_names
    rows = []
    for dataset_name in dataset_names:
        dataset_dir, summary_rows = load_dataset_summary(dataset_name)
        aggregate = aggregate_summary_rows(summary_rows)
        official_mae = OFFICIAL_TRANSFER_MAE[dataset_name]
        rows.append(
            {
                "dataset": dataset_name,
                "result_dir": str(dataset_dir.relative_to(REPO_ROOT)),
                "official_mae": official_mae,
                "our_mean_mae": aggregate["mean_test_mae"],
                "our_std_mae": aggregate["std_test_mae"],
                "mae_gap": aggregate["mean_test_mae"] - official_mae,
                "our_mean_r2": aggregate["mean_test_r2"],
                "rows_averaged": aggregate["num_rows"],
                "mean_test_size": aggregate["mean_test_size"],
            }
        )
    return rows


def maybe_frame(records):
    if pd is None:
        return records
    return pd.DataFrame(records)


SMRT_RUN = latest_stage_run("SMRT")
RIKEN_RUN = latest_stage_run("RIKENMONA")

print(f"Repo root: {REPO_ROOT}")
print(f"Latest SMRT run: {SMRT_RUN.name}")
print(f"Latest RIKENMONA run: {RIKEN_RUN.name}")
print(f"Transfer datasets found: {len(TRANSFER_ORDER)}")

In [ ]:
RUN_PATH = SMRT_RUN
DATASET_NAME = "Eawag_XBridgeC18_364"

print("Selected run:", RUN_PATH)
print("Selected dataset:", DATASET_NAME)
print("Dataset results directory:", find_dataset_dir(DATASET_NAME))

In [ ]:
def plot_single_run(run_path):
    run_path = Path(run_path)
    log_rows = normalize_rows(read_csv_rows(run_path / "log.csv"))
    y, p = read_predictions(run_path / "test_predictions.csv")
    test_metrics = compute_metrics(y, p)
    best_row = min(log_rows, key=lambda row: float(row["valid_loss"]))

    epochs = [int(row["epoch"]) for row in log_rows]
    train_loss = [float(row["train_loss"]) for row in log_rows]
    valid_loss = [float(row["valid_loss"]) for row in log_rows]
    test_loss = [float(row["test_loss"]) for row in log_rows]
    train_mae = [float(row["train_mae"]) for row in log_rows]
    valid_mae = [float(row["valid_mae"]) for row in log_rows]
    test_mae = [float(row["test_mae"]) for row in log_rows]

    fig, axes = plt.subplots(2, 2, figsize=(12, 9))

    axes[0, 0].plot(epochs, train_loss, label="train")
    axes[0, 0].plot(epochs, valid_loss, label="valid")
    axes[0, 0].plot(epochs, test_loss, label="test")
    axes[0, 0].axvline(best_row["epoch"], color="black", linestyle="--", alpha=0.6)
    axes[0, 0].set_title("Loss Curves")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].legend()

    axes[0, 1].plot(epochs, train_mae, label="train", color="tab:blue")
    axes[0, 1].plot(epochs, valid_mae, label="valid", color="tab:orange")
    axes[0, 1].plot(epochs, test_mae, label="test", color="tab:green")
    axes[0, 1].axvline(best_row["epoch"], color="black", linestyle="--", alpha=0.6)
    axes[0, 1].set_title("MAE Curves")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("MAE")
    axes[0, 1].legend()

    axes[1, 0].plot(epochs, valid_loss, color="tab:red", label="valid_loss")
    axes[1, 0].plot(epochs, valid_mae, color="tab:purple", label="valid_mae")
    axes[1, 0].axvline(best_row["epoch"], color="black", linestyle="--", alpha=0.6)
    axes[1, 0].set_title("Validation Selection Signals")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].legend()

    axes[1, 1].scatter(y, p, alpha=0.55, s=18)
    lo = min(min(y), min(p))
    hi = max(max(y), max(p))
    axes[1, 1].plot([lo, hi], [lo, hi], linestyle="--", color="black")
    axes[1, 1].set_title(f"Test Prediction Scatter | MAE={test_metrics['mae']:.2f} s")
    axes[1, 1].set_xlabel("Target RT (s)")
    axes[1, 1].set_ylabel("Predicted RT (s)")

    fig.suptitle(run_path.name, fontsize=14)
    fig.tight_layout()
    plt.show()
    return best_row, test_metrics


best_row, run_metrics = plot_single_run(RUN_PATH)
pprint({
    "run": RUN_PATH.name,
    "best_epoch": best_row["epoch"],
    "best_valid_loss": best_row["valid_loss"],
    **run_metrics,
})

if pd is not None:
    pd.DataFrame([best_row])

In [ ]:
dataset_dir, summary_rows = load_dataset_summary(DATASET_NAME)
aggregate = aggregate_summary_rows(summary_rows)
comparison = {
    "dataset": DATASET_NAME,
    "result_dir": str(dataset_dir.relative_to(REPO_ROOT)),
    "official_mae": OFFICIAL_TRANSFER_MAE.get(DATASET_NAME),
    "our_mean_mae": aggregate["mean_test_mae"],
    "our_std_mae": aggregate["std_test_mae"],
    "mae_gap": None if OFFICIAL_TRANSFER_MAE.get(DATASET_NAME) is None else aggregate["mean_test_mae"] - OFFICIAL_TRANSFER_MAE[DATASET_NAME],
    "our_mean_r2": aggregate["mean_test_r2"],
    "rows_averaged": aggregate["num_rows"],
    "mean_test_size": aggregate["mean_test_size"],
    "min_best_epoch": aggregate["min_best_epoch"],
    "max_best_epoch": aggregate["max_best_epoch"],
}

pprint(comparison)

folds = [row.get("fold") for row in summary_rows]
maes = [float(row["test_mae"]) for row in summary_rows]
fig, ax = plt.subplots(figsize=(7.4, 4.6))
ax.plot(range(1, len(maes) + 1), maes, marker="o", linewidth=1.6)
ax.axhline(comparison["official_mae"], color="black", linestyle="--", label="official MAE")
ax.axhline(comparison["our_mean_mae"], color="tab:red", linestyle=":", label="our mean MAE")
ax.set_title(f"{DATASET_NAME}: fold-level test MAE")
ax.set_xlabel("Fold row index")
ax.set_ylabel("Test MAE (s)")
ax.legend()
fig.tight_layout()
plt.show()

if pd is not None:
    pd.DataFrame(summary_rows)

In [ ]:
overview = build_transfer_overview()
overview_frame = maybe_frame(overview)
overview_frame

In [ ]:
overview = build_transfer_overview()
labels = [row["dataset"].replace("_", "\n") for row in overview]
official = [row["official_mae"] for row in overview]
ours = [row["our_mean_mae"] for row in overview]
ours_std = [row["our_std_mae"] for row in overview]
positions = list(range(len(overview)))
width = 0.38

fig, ax = plt.subplots(figsize=(12.8, 5.8))
ax.bar([x - width / 2 for x in positions], official, width=width, color="#9aa0a6", label="Official mean MAE")
ax.bar([x + width / 2 for x in positions], ours, width=width, yerr=ours_std, capsize=3, color="#1f77b4", label="Our mean MAE (summary.csv rows)")
ax.axvline(len(PAPER_SEVEN) - 0.5, color="black", linestyle=":", linewidth=1.0)
ax.set_xticks(positions)
ax.set_xticklabels(labels)
ax.set_ylabel("MAE (s)")
ax.set_title("Transfer-learning overview with the paper-style aggregation rule")
ax.legend(frameon=True)
fig.tight_layout()
plt.show()

seven = [row for row in overview if row["dataset"] in PAPER_SEVEN]
all_ten = overview
print("Seven paper datasets | official mean MAE:", round(sum(row["official_mae"] for row in seven) / len(seven), 2))
print("Seven paper datasets | our mean MAE:", round(sum(row["our_mean_mae"] for row in seven) / len(seven), 2))
print("All ten datasets | official mean MAE:", round(sum(row["official_mae"] for row in all_ten) / len(all_ten), 2))
print("All ten datasets | our mean MAE:", round(sum(row["our_mean_mae"] for row in all_ten) / len(all_ten), 2))